# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [1]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# List to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None  # Default to None (indicating missing content)
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)  # Log failed download

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 467
n_rows = 20 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"], 
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")


/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Fetching .mod files:  15%|█▌        | 3/20 [00:00<00:01, 10.72it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod


Fetching .mod files: 100%|██████████| 20/20 [00:01<00:00, 11.60it/s]

⚠️ Some downloads failed. Check ../data/failed_downloads.json for details.

✅ All .mod files (including failures) saved in ../data/mod_files.json


# Read JSON file

In [2]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

In [3]:
def View(df, rows=5, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [4]:
# Function to extract mod directory from the URL
def extract_mod_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def extract_mod_name(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def extract_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to determine exclusion reason with snake_case labels
def determine_exclusion(row):
    if pd.isna(row["content"]):
        return "not_found"  # Standardized exclusion for missing files
    if "x86_64" in row["url"]:
        return "x86_64"  # Standardized exclusion for architecture issues
    return None  # Keep valid entries as None (not excluded)

# Function to extract all TITLE occurrences from .mod content
def extract_title(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"^TITLE\s+(.+)", content, re.MULTILINE)  # Find all TITLE lines
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all COMMENT sections from .mod content
def extract_comment(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"COMMENT\s+(.*?)\s+ENDCOMMENT", content, re.DOTALL)  # Capture all COMMENT sections
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all SUFFIX occurrences from .mod content
def extract_mod_suffix(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"SUFFIX\s+(\w+)", content)  # Find all SUFFIX statements
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all USEION occurrences from .mod content
def extract_mod_use_ion(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"USEION\s+(\w+)", content)  # Find all USEION statements
    return matches if matches else None  # Return list if multiple, else None

# Function to extract webpage heading
def extract_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def extract_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def extract_first_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

def extract_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)\d{2}\b", citation)  # Find a 4-digit year (1900-2099)
    return match.group(0) if match else None  # Extract the year

In [5]:
# Load JSON into DataFrame
df = pd.read_json("mod_files.json")

# Set "row_id" as the index
df.set_index("row_id", inplace=True)

# Exclude mod files
df["mod_exclude"] = df.apply(determine_exclusion, axis=1)  # Apply exclusion function

# Extract features from url
df["mod_heading"] = df["url"].apply(extract_heading)  # Extract heading from webpage
df["mod_citation"] = df["mod_heading"].apply(extract_citation)
df["mod_first_author"] = df["mod_citation"].apply(extract_first_author)
df["mod_year"] = df["mod_citation"].apply(extract_year)  # Now handles multiple years


df["mod_dir"] = df["url"].apply(extract_mod_dir)
df["mod_name"] = df["url"].apply(extract_mod_name)
df["mod_model_id"] = df["url"].apply(extract_model_id)

#  Extract features from content
df["mod_title"] = df["content"].apply(extract_title)
df["mod_comment"] = df["content"].apply(extract_comment)
df["mod_suffix"] = df["content"].apply(extract_mod_suffix)
df["mod_use_ion"] = df["content"].apply(extract_mod_use_ion)



FileNotFoundError: File mod_files.json does not exist

In [ ]:
! git add .
! git commit -m "deleted v1 download version"
! git push 

In [ ]:
!pwd

In [ ]:
View(df,3)

In [ ]:
sample = """ 
TITLE K-Cl cotransporter KCC2

INDEPENDENT {t FROM 0 TO 1 WITH 10 (ms)}

NEURON {
	SUFFIX kcc2
	USEION k READ ko, ki WRITE ik
	USEION cl READ clo, cli WRITE icl VALENCE -1
	RANGE ik, icl
	GLOBAL U, S, Vi
}

UNITS {
	(mV)	= (millivolt)
	(molar) = (1/liter)
	(mM)	= (millimolar)
	(um)	= (micron)
	(mA)	= (milliamp)
	(mol)	= (1)
	FARADAY	= 96485.309 (coul/mole)
	PI	= (pi) (1)
}

PARAMETER {
	S = 5654.87 (um2)
  	Vi = 8913.48 (um3)
	U = 0.0003    (mM/ms)
}

ASSIGNED {
	ik		(mA/cm2)
	icl		(mA/cm2)
	ko		(mM)
	ki		(mM)
  	clo   (mM)
  	cli   (mM)
}

BREAKPOINT {
	LOCAL rate
	rate = pumprate(ki,ko,cli,clo)
	ik =  rate
	icl = -rate
}

FUNCTION pumprate(ki,ko,cli,clo) {
	pumprate = U*log((ki*cli)/(ko*clo))*(FARADAY*Vi/(S*1e4))
}
"""

In [ ]:
extract_mod_use_ion(sample)

In [ ]:
! git push